In [1]:
import pandas as pd
import altair as alt

In [2]:
eudf = pd.read_csv('../data/fully_cleaned_top_spotify_songs_EU.csv')
eudf

,artists,country,popularity,is_explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,duration_s,artist_count,artists_list
0,"tjuvjakt, fanny avonne",SE,76,False,0.735,0.6530,3,-6.261,1,0.0533,0.3600,0.000000,0.0791,0.620,116.002,4,190.344,2,"['tjuvjakt', 'fanny avonne']"
1,ida-lova,SE,77,False,0.404,0.6830,1,-6.969,1,0.0594,0.3020,0.000000,0.3180,0.661,162.091,4,187.076,1,['ida-lova']
2,lov1,SE,64,False,0.695,0.6760,8,-6.859,1,0.0384,0.1010,0.000000,0.0929,0.431,121.962,4,208.201,1,['lov1']
3,kaj,SE,75,False,0.757,0.8890,9,-4.729,0,0.0788,0.0754,0.000023,0.0397,0.754,106.037,4,166.880,1,['kaj']
4,"humlan djojj, josefine götestam",SE,76,False,0.684,0.0326,0,-17.898,1,0.0583,0.9730,0.000002,0.1080,0.440,66.685,5,140.046,2,"['humlan djojj', 'josefine götestam']"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,chappell roan,GB,48,False,0.742,0.7570,6,-4.981,1,0.0421,0.0187,0.000000,0.3050,0.957,139.982,4,184.841,1,['chappell roan']
4996,central cee,GB,36,True,0.750,0.7370,11,-4.245,0,0.3000,0.4450,0.000000,0.0772,0.962,142.402,4,198.400,1,['central cee']
4997,myles smith,GB,75,False,0.762,0.7830,5,-5.206,0,0.0387,0.3290,0.000000,0.0536,0.840,115.091,4,176.000,1,['myles smith']
4998,"pawsa, the adventures of stevie v",GB,81,False,0.783,0.4730,0,-6.493,1,0.1670,0.0693,0.000034,0.1490,0.539,131.890,4,221.448,2,"['pawsa', 'the adventures of stevie v']"


In [9]:
audio_features = [
    'danceability', 'energy', 'speechiness',
    'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo', 'duration_s'
]

df_long = eudf.melt(
    id_vars=['country'],
    value_vars=audio_features,
    var_name='feature',
    value_name='value'
)

df_grouped = df_long.groupby(['country', 'feature']).agg(
    value=('value', 'mean')
).reset_index()

df_long

,country,feature,value
0,SE,danceability,0.735
1,SE,danceability,0.404
2,SE,danceability,0.695
3,SE,danceability,0.757
4,SE,danceability,0.684
...,...,...,...
44995,GB,duration_s,184.841
44996,GB,duration_s,198.400
44997,GB,duration_s,176.000
44998,GB,duration_s,221.448


In [34]:
# # original interactive barchart, altair
# feature_radio = alt.binding_radio(
#     options=audio_features,
#     name='Audio Feature: '
# )

# selection = alt.selection_point(
#     fields=['feature'],
#     bind=feature_radio,
#     value=[{'feature': 'danceability'}]  # default
# )

# chart = alt.Chart(df_grouped).mark_bar().encode(
#     x=alt.X('country:N', title='Country'),
#     y=alt.Y('value:Q', title='Value'),
#     color=alt.Color('country:N'),
#     opacity = alt.OpacityValue(0.5),
#     tooltip=[
#         alt.Tooltip('country:N'),
#         alt.Tooltip('mean(value):Q', title='Avg Value')
#     ]
# ).properties(
#     title='Country Comparison of Average Audio Features',
#     width=600,
#     height=400
# ).add_params(
#     selection
# ).transform_filter(
#     selection
# )

# chart

In [21]:
agg_options = ['mean', 'median', 'mode']

agg_radio = alt.binding_radio(
    options=agg_options,
    name='Aggregation: '
)

agg_selection = alt.selection_point(
    fields=['aggregation'],
    bind=agg_radio,
    value=[{'aggregation': 'mean'}]
)

In [36]:
def mode_with_count(x):
    m = x.mode()
    if len(m) == 0:
        return pd.Series([None, 0])
    
    mode_val = m.iloc[0]
    count = (x == mode_val).sum()
    return pd.Series([mode_val, count])

df_agg = df_long.groupby(['country', 'feature']).agg(
    mean=('value', 'mean'),
    median=('value', 'median')
)

# mode must be added separately (since it returns 2 values)
mode_df = df_long.groupby(['country', 'feature'])['value'].apply(mode_with_count).unstack()
mode_df.columns = ['mode', 'mode_count']

df_agg = df_agg.join(mode_df).reset_index()

df_agg_long = df_agg.melt(
    id_vars=['country', 'feature', 'mode_count'],
    value_vars=['mean', 'median', 'mode'],
    var_name='aggregation',
    value_name='value'
)

chart = alt.Chart(df_agg_long).mark_bar().encode(
    x=alt.X('country:N', title='Country'),
    y=alt.Y('value:Q', title='Value'),
    color=alt.Color('country:N'),
    opacity = alt.OpacityValue(0.5),
    tooltip=[
        alt.Tooltip('country:N'),
        alt.Tooltip('aggregation:N'),
        alt.Tooltip('value:Q', title='Value'),
        alt.Tooltip('mode_count:Q', title='Mode Count')
    ]
).properties(
    title='Country Comparison of Average Audio Features',
    width=600,
    height=400
).add_params(
    selection,        # feature selector
    agg_selection     # aggregation selector
).transform_filter(
    selection
).transform_filter(
    agg_selection
)

chart

alt.Chart(...)

In [38]:
chart.save("barchart.html")